In [ ]:
!pip install -q statsmodels
import warnings
warnings.filterwarnings('ignore')
print('statsmodels ready.')


In [ ]:
import os
import json
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/PACL_Project"
PHASE3_DIR  = f"{PROJECT_DIR}/phase3_outputs"
PHASE4_DIR  = f"{PROJECT_DIR}/phase4_outputs"
os.makedirs(PHASE4_DIR, exist_ok=True)

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_records     = load_jsonl(f"{PHASE3_DIR}/train.jsonl")
test_open_records = load_jsonl(f"{PHASE3_DIR}/test_open.jsonl")
train_poets    = [r['poet']    for r in train_records]
train_poem_ids = [r['poem_id'] for r in train_records]
train_texts    = [r['text']    for r in train_records]
print(f"train: {len(train_records):,} | test_open: {len(test_open_records):,}")


In [ ]:
import numpy as np

train_emb     = np.load(f"{PHASE3_DIR}/train_embeddings_byt5base.npy")
test_open_emb = np.load(f"{PHASE3_DIR}/test_open_embeddings_byt5base.npy")
print(f"train_emb: {train_emb.shape} | test_open_emb: {test_open_emb.shape}")
assert not np.isnan(train_emb).any() and not np.isnan(test_open_emb).any()
print('OK - no NaN')


In [ ]:
import time
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import defaultdict

vectorizer = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2, 5),
    max_features=50000, sublinear_tf=True, dtype=np.float32
)
vectorizer.fit(train_texts)
print(f"TF-IDF fit: {len(vectorizer.vocabulary_):,} features")

K_VALUES = [1, 6]

def get_k(rec):
    try:
        return int(rec.get('k', rec.get('num_shots', -1)))
    except Exception:
        return -1

test_open_by_k = defaultdict(lambda: {'records': [], 'indices': []})
for i, r in enumerate(test_open_records):
    k = get_k(r)
    if k in K_VALUES:
        test_open_by_k[k]['records'].append(r)
        test_open_by_k[k]['indices'].append(i)

open_emb_by_k   = {}
open_tfidf_by_k = {}
for k in K_VALUES:
    idxs = test_open_by_k[k]['indices']
    recs = test_open_by_k[k]['records']
    open_emb_by_k[k]   = test_open_emb[idxs]
    open_tfidf_by_k[k] = vectorizer.transform([r['text'] for r in recs])
    print(f"k={k}: {len(recs):,} recs | emb={open_emb_by_k[k].shape} | tfidf={open_tfidf_by_k[k].shape}")
print('group_by_k done.')


In [ ]:
import numpy as np
import scipy.sparse as sp
from sklearn.preprocessing import normalize as sk_normalize

def l2_normalize(mat):
    if sp.issparse(mat):
        return sk_normalize(mat, norm='l2', axis=1)
    mat = np.asarray(mat, dtype=np.float32)
    norms = np.linalg.norm(mat, axis=1, keepdims=True)
    return mat / (norms + 1e-12)

def compute_sims(bq, g):
    sims = bq @ g.T
    if sp.issparse(sims):
        sims = sims.toarray()
    return np.asarray(sims, dtype=np.float32)

def sort_top(sims, depth):
    d = min(depth, sims.shape[1])
    top_idx = np.argpartition(-sims, kth=d - 1, axis=1)[:, :d]
    ri = np.arange(top_idx.shape[0])[:, None]
    top_s = sims[ri, top_idx]
    order = np.argsort(-top_s, axis=1)
    return top_idx[ri, order]

def eval_per_query(query_vecs, query_poets, query_poem_ids,
                   gallery_vecs, gallery_poets, gallery_poem_ids,
                   exclude_same_poem=True, batch_size=500, mrr_depth=100):
    q  = l2_normalize(query_vecs)
    g  = l2_normalize(gallery_vecs)
    qp = np.asarray(query_poets)
    gp = np.asarray(gallery_poets)
    qi = np.asarray(query_poem_ids)
    gi = np.asarray(gallery_poem_ids)
    N  = q.shape[0]
    hits1_list = []
    rr_list    = []
    for start in range(0, N, batch_size):
        end  = min(start + batch_size, N)
        sims = compute_sims(q[start:end], g)
        if exclude_same_poem:
            mask = qi[start:end][:, None] == gi[None, :]
            sims = np.where(mask, -np.inf, sims)
        top_idx = sort_top(sims, mrr_depth)
        bp = qp[start:end]
        for i in range(end - start):
            retrieved = gp[top_idx[i]]
            hp = np.where(retrieved == bp[i])[0]
            if len(hp) > 0:
                rank = int(hp[0]) + 1
                hits1_list.append(1 if rank == 1 else 0)
                rr_list.append(1.0 / rank)
            else:
                hits1_list.append(0)
                rr_list.append(0.0)
    return np.array(hits1_list, dtype=np.int8), np.array(rr_list, dtype=np.float32)

def eval_rrf_per_query(dense_q, dense_g, sparse_q, sparse_g,
                       query_poets, query_poem_ids,
                       gallery_poets, gallery_poem_ids,
                       mu=1.0, exclude_same_poem=True,
                       batch_size=200, c_depth=500, mrr_depth=100):
    dq = l2_normalize(dense_q)
    dg = l2_normalize(dense_g)
    sq = l2_normalize(sparse_q)
    sg = l2_normalize(sparse_g)
    qp = np.asarray(query_poets)
    gp = np.asarray(gallery_poets)
    qi = np.asarray(query_poem_ids)
    gi = np.asarray(gallery_poem_ids)
    N  = dq.shape[0]
    hits1_list = []
    rr_list    = []
    for start in range(0, N, batch_size):
        end   = min(start + batch_size, N)
        dsims = compute_sims(dq[start:end], dg)
        ssims = compute_sims(sq[start:end], sg)
        if exclude_same_poem:
            mask  = qi[start:end][:, None] == gi[None, :]
            dsims = np.where(mask, -np.inf, dsims)
            ssims = np.where(mask, -np.inf, ssims)
        d_idx = sort_top(dsims, c_depth)
        s_idx = sort_top(ssims, c_depth)
        bp = qp[start:end]
        for i in range(end - start):
            scores = defaultdict(float)
            for rank, giv in enumerate(d_idx[i].tolist(), 1):
                scores[giv] += 1.0 / (60 + rank)
            for rank, giv in enumerate(s_idx[i].tolist(), 1):
                scores[giv] += mu / (60 + rank)
            if not scores:
                hits1_list.append(0)
                rr_list.append(0.0)
                continue
            sorted_gi = sorted(scores, key=lambda x: -scores[x])[:mrr_depth]
            retrieved = gp[sorted_gi]
            hp = np.where(retrieved == bp[i])[0]
            if len(hp) > 0:
                rank = int(hp[0]) + 1
                hits1_list.append(1 if rank == 1 else 0)
                rr_list.append(1.0 / rank)
            else:
                hits1_list.append(0)
                rr_list.append(0.0)
    return np.array(hits1_list, dtype=np.int8), np.array(rr_list, dtype=np.float32)

print('eval_per_query and eval_rrf_per_query ready. mrr_depth=100')


In [ ]:
byt5_hits = {}
byt5_rr   = {}

for k in K_VALUES:
    recs       = test_open_by_k[k]['records']
    q_poets    = [r['poet']    for r in recs]
    q_poem_ids = [r['poem_id'] for r in recs]
    print(f"ByT5 Open k={k} ({len(recs):,} queries) ...")
    t0 = time.time()
    h, rr = eval_per_query(
        open_emb_by_k[k], q_poets, q_poem_ids,
        open_emb_by_k[k], q_poets, q_poem_ids,
        exclude_same_poem=True, batch_size=500, mrr_depth=100
    )
    byt5_hits[k] = h
    byt5_rr[k]   = rr
    r1    = float(h.mean())
    mrr_v = float(rr.mean())
    print(f"  Recall@1={r1:.4f} | MRR={mrr_v:.4f} | {time.time()-t0:.1f}s")
    ref = {1: 0.3157, 6: 0.6233}[k]
    print(f"  Phase4.1 ref={ref} | {'OK' if abs(r1 - ref) < 0.002 else 'MISMATCH!'}")

print('ByT5 done.')


In [ ]:
tfidf_hits = {}
tfidf_rr   = {}

for k in K_VALUES:
    recs       = test_open_by_k[k]['records']
    q_poets    = [r['poet']    for r in recs]
    q_poem_ids = [r['poem_id'] for r in recs]
    print(f"TF-IDF Open k={k} ({len(recs):,} queries) ...")
    t0 = time.time()
    h, rr = eval_per_query(
        open_tfidf_by_k[k], q_poets, q_poem_ids,
        open_tfidf_by_k[k], q_poets, q_poem_ids,
        exclude_same_poem=True, batch_size=500, mrr_depth=100
    )
    tfidf_hits[k] = h
    tfidf_rr[k]   = rr
    r1    = float(h.mean())
    mrr_v = float(rr.mean())
    print(f"  Recall@1={r1:.4f} | MRR={mrr_v:.4f} | {time.time()-t0:.1f}s")
    ref = {1: 0.3576, 6: 0.5302}[k]
    print(f"  Phase4.1 ref={ref} | {'OK' if abs(r1 - ref) < 0.003 else 'MISMATCH!'}")

print('TF-IDF done.')


In [ ]:
BEST_MU = 1.0
C_DEPTH = 500

rrf_hits = {}
rrf_rr   = {}

for k in K_VALUES:
    recs       = test_open_by_k[k]['records']
    q_poets    = [r['poet']    for r in recs]
    q_poem_ids = [r['poem_id'] for r in recs]
    print(f"RRF Open k={k} (mu={BEST_MU}, c_depth={C_DEPTH}) -- may take a few minutes")
    t0 = time.time()
    h, rr = eval_rrf_per_query(
        open_emb_by_k[k],   open_emb_by_k[k],
        open_tfidf_by_k[k], open_tfidf_by_k[k],
        q_poets, q_poem_ids, q_poets, q_poem_ids,
        mu=BEST_MU, exclude_same_poem=True,
        batch_size=200, c_depth=C_DEPTH, mrr_depth=100
    )
    rrf_hits[k] = h
    rrf_rr[k]   = rr
    r1    = float(h.mean())
    mrr_v = float(rr.mean())
    print(f"  Recall@1={r1:.4f} | MRR={mrr_v:.4f} | {time.time()-t0:.1f}s")
    ref = {1: 0.3900, 6: 0.7054}[k]
    print(f"  Phase4.1 ref={ref} | {'OK' if abs(r1 - ref) < 0.003 else 'MISMATCH!'}")

np.savez_compressed(
    f"{PHASE4_DIR}/step1_byt5_per_query.npz",
    hits_k1=byt5_hits[1], rr_k1=byt5_rr[1],
    hits_k6=byt5_hits[6], rr_k6=byt5_rr[6]
)
np.savez_compressed(
    f"{PHASE4_DIR}/step1_tfidf_per_query.npz",
    hits_k1=tfidf_hits[1], rr_k1=tfidf_rr[1],
    hits_k6=tfidf_hits[6], rr_k6=tfidf_rr[6]
)
np.savez_compressed(
    f"{PHASE4_DIR}/step1_rrf_per_query.npz",
    hits_k1=rrf_hits[1], rr_k1=rrf_rr[1],
    hits_k6=rrf_hits[6], rr_k6=rrf_rr[6]
)
print("3 npz files saved to Drive.")


In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

def compute_mcnemar(correct_A, correct_B):
    ca = correct_A.astype(bool)
    cb = correct_B.astype(bool)
    both_correct = int(np.sum(ca & cb))
    A_only       = int(np.sum(ca & ~cb))
    B_only       = int(np.sum(~ca & cb))
    both_wrong   = int(np.sum(~ca & ~cb))
    table = [[both_correct, A_only],
             [B_only,       both_wrong]]
    result = mcnemar(table, exact=False, correction=True)
    return float(result.statistic), float(result.pvalue)

print('=' * 60)
print('McNemar Test: RRF vs TF-IDF  and  RRF vs ByT5')
print('=' * 60)

mcnemar_results = {}
for k in K_VALUES:
    print(f'k={k}:')
    stat1, p1 = compute_mcnemar(rrf_hits[k], tfidf_hits[k])
    stat2, p2 = compute_mcnemar(rrf_hits[k], byt5_hits[k])
    sig1 = 'YES (p<0.05)' if p1 < 0.05 else 'NO'
    sig2 = 'YES (p<0.05)' if p2 < 0.05 else 'NO'
    print(f'  RRF vs TF-IDF: stat={stat1:.2f}  p={p1:.4e}  sig={sig1}')
    print(f'  RRF vs ByT5:   stat={stat2:.2f}  p={p2:.4e}  sig={sig2}')
    mcnemar_results[k] = {
        'rrf_vs_tfidf': {'stat': stat1, 'pvalue': p1, 'significant': p1 < 0.05},
        'rrf_vs_byt5':  {'stat': stat2, 'pvalue': p2, 'significant': p2 < 0.05},
    }

print('McNemar done.')


In [ ]:
B_ITER = 10_000
RNG    = np.random.default_rng(42)

def bootstrap_ci(arr, B=B_ITER, alpha=0.05):
    n   = len(arr)
    idx = RNG.integers(0, n, size=(B, n))
    means = arr.astype(np.float64)[idx].mean(axis=1)
    lo = float(np.percentile(means, 100.0 * alpha / 2.0))
    hi = float(np.percentile(means, 100.0 * (1.0 - alpha / 2.0)))
    return lo, hi

print(f'Paired Bootstrap CI  B={B_ITER:,}  seed=42')
print('=' * 60)

ci_results = {}
for k in K_VALUES:
    print(f'k={k}:')
    ci_results[k] = {}
    for name, h_arr, rr_arr in [
        ('ByT5',   byt5_hits[k],  byt5_rr[k]),
        ('TF-IDF', tfidf_hits[k], tfidf_rr[k]),
        ('RRF',    rrf_hits[k],   rrf_rr[k]),
    ]:
        r1_val   = float(h_arr.mean())
        mrr_val  = float(rr_arr.mean())
        r1_lo,  r1_hi  = bootstrap_ci(h_arr.astype(np.float32))
        mrr_lo, mrr_hi = bootstrap_ci(rr_arr)
        print(
            f'  {name:<8} Recall@1={r1_val:.4f} [{r1_lo:.4f}, {r1_hi:.4f}]  '
            f'MRR={mrr_val:.4f} [{mrr_lo:.4f}, {mrr_hi:.4f}]'
        )
        ci_results[k][name] = {
            'recall1':    r1_val,
            'recall1_lo': r1_lo,
            'recall1_hi': r1_hi,
            'mrr':        mrr_val,
            'mrr_lo':     mrr_lo,
            'mrr_hi':     mrr_hi,
        }

print('Bootstrap done.')


In [ ]:
import json as _json

rows = []
for k in K_VALUES:
    rrf_r1     = ci_results[k]['RRF']['recall1']
    rrf_r1_lo  = ci_results[k]['RRF']['recall1_lo']
    rrf_r1_hi  = ci_results[k]['RRF']['recall1_hi']
    rrf_mrr    = ci_results[k]['RRF']['mrr']
    rrf_mrr_lo = ci_results[k]['RRF']['mrr_lo']
    rrf_mrr_hi = ci_results[k]['RRF']['mrr_hi']
    for baseline_name in ['TF-IDF', 'ByT5']:
        mc_key  = 'rrf_vs_tfidf' if baseline_name == 'TF-IDF' else 'rrf_vs_byt5'
        pval    = mcnemar_results[k][mc_key]['pvalue']
        sig_str = 'Yes' if pval < 0.05 else 'No'
        pval_s  = f'{pval:.2e}'
        base_r1  = ci_results[k][baseline_name]['recall1']
        gain_r1  = rrf_r1 - base_r1
        ci_r1_s  = f'[{rrf_r1_lo:.4f}, {rrf_r1_hi:.4f}]'
        rows.append(
            f'  RRF vs {baseline_name} ($k={k}$) & Recall@1 & '
            f'{base_r1:.4f} & {rrf_r1:.4f} & {gain_r1:+.4f} & '
            f'{ci_r1_s} & {pval_s} & {sig_str} \\\\'
        )
        base_mrr  = ci_results[k][baseline_name]['mrr']
        gain_mrr  = rrf_mrr - base_mrr
        ci_mrr_s  = f'[{rrf_mrr_lo:.4f}, {rrf_mrr_hi:.4f}]'
        rows.append(
            f'  RRF vs {baseline_name} ($k={k}$) & MRR & '
            f'{base_mrr:.4f} & {rrf_mrr:.4f} & {gain_mrr:+.4f} & '
            f'{ci_mrr_s} & --- & --- \\\\'
        )

latex_parts = [
    '\\begin{table}[t]',
    '\\centering',
    '\\caption{Statistical Significance of RRF vs. Baselines on Open-Set (Zero-Shot).}',
    '\\label{tab:significance}',
    '\\begin{tabular}{llcccccc}',
    '\\toprule',
    '\\textbf{Comparison} & \\textbf{Metric} & \\textbf{Baseline} & \\textbf{RRF} & \\textbf{Gain} & \\textbf{95\\% CI (RRF)} & $p$-value & Sig. \\\\',
    '\\midrule',
] + rows + ['\\bottomrule', '\\end{tabular}', '\\end{table}']
latex_table = '\n'.join(latex_parts)

print('=== LaTeX Table ===')
print(latex_table)

out = {
    'experiment': 'Exp1_StatisticalSignificance',
    'spec_source': 'EXP.txt (professor instructions)',
    'best_mu_rrf': BEST_MU,
    'c_depth': C_DEPTH,
    'mrr_depth': 100,
    'bootstrap_B': B_ITER,
    'mcnemar_results': mcnemar_results,
    'ci_results': ci_results,
    'latex_table': latex_table,
}
out_path = f'{PHASE4_DIR}/exp1_significance_results.json'
with open(out_path, 'w', encoding='utf-8') as f:
    _json.dump(out, f, ensure_ascii=False, indent=2)
print(f'Saved: {out_path}')
